# Project - Multi-modal Airline AI Assistant (Open Source edition)

This is `week2/day5.ipynb` with every paid API swapped for something open source and local. Nothing here calls OpenAI or Anthropic, and nothing costs a cent per run.

| day5 uses | this notebook uses |
|---|---|
| `gpt-4.1-mini` for chat + tools | **`llama3.2`** via Ollama's OpenAI-compatible endpoint |
| `gpt-image-1-mini` for images (~3c each) | **`stabilityai/sd-turbo`** via `diffusers`, running locally |
| `gpt-4o-mini-tts` for speech | **Windows SAPI**, offline and built in |

**The image caveat, up front:** this machine has no CUDA GPU, so Stable Diffusion runs on CPU at roughly 30-90 seconds per image - slow enough to make the chat feel broken. So image generation is **off by default**, behind the `ENABLE_IMAGES` flag in the config cell. With it off you still get a generated placeholder card, so the UI works end to end. Flip it on when you want to watch real diffusion happen.

**Prerequisites:** Ollama running with `ollama pull llama3.2`. For images only, additionally `uv add diffusers accelerate` (or `pip install diffusers accelerate`).

In [1]:
# imports

import os
import sys
import json
import sqlite3
import tempfile
import subprocess
from itertools import count

import requests
from openai import OpenAI
import gradio as gr
from PIL import Image, ImageDraw, ImageFont

## Initialization

Change `OLLAMA_PORT` if your Ollama listens somewhere other than the default 11434.

In [2]:
OLLAMA_PORT = 11434
OLLAMA_HOST = f"http://localhost:{OLLAMA_PORT}"
MODEL = "llama3.2"

DB = "prices.db"
MAX_TOOL_ROUNDS = 5

# Real Stable Diffusion is slow on a CPU-only machine. Off by default - see the notes above.
ENABLE_IMAGES = False

# No API key needed - Ollama ignores it, but the openai client insists on one being present
openai = OpenAI(base_url=f"{OLLAMA_HOST}/v1", api_key="ollama")

In [ ]:
# Sanity check: is Ollama up, and does our model exist and support tools?

try:
    models = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5).json()["models"]
except Exception as e:
    raise SystemExit(f"Could not reach Ollama at {OLLAMA_HOST} - is it running? ({e})")

print(f"Ollama is up at {OLLAMA_HOST}. Models available:")
for m in models:
    tools_ok = "tools" in m.get("capabilities", [])
    print(f"  {m['name']:<20} tools: {'yes' if tools_ok else 'no'}")

names = [m["name"] for m in models]
if not any(n == MODEL or n.startswith(f"{MODEL}:") for n in names):
    print(f"\nWARNING: '{MODEL}' not found. Run:  ollama pull {MODEL}")

In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.

You do not know any ticket prices from memory. Whenever a customer asks about the price
of a ticket, you must call the get_ticket_price tool to look it up. Never guess a price.
Each call handles exactly one city, so if the customer asks about several cities,
call the tool once for each of them.
"""

## The database and our tool

day5 assumes `prices.db` already exists from day4. We create and seed it here so this notebook stands alone.

In [ ]:
with sqlite3.connect(DB) as conn:
    conn.execute("CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)")
    conn.commit()


def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT price FROM prices WHERE city = ?", (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"


def set_ticket_price(city, price):
    price = float(price)
    with sqlite3.connect(DB) as conn:
        conn.execute(
            "INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?",
            (city.lower(), price, price)
        )
        conn.commit()
    return f"Ticket price to {city} is now set to ${price}"


# Seed, so the notebook works even on a fresh machine
for city, price in {"london": 799, "paris": 899, "tokyo": 1420, "sydney": 2999, "berlin": 499}.items():
    set_ticket_price(city, price)

In [ ]:
get_ticket_price("Paris")

In [ ]:
# The tool schema. The wording is blunter than the frontier version needs - at 3B,
# without "one city, never a list", llama3.2 passes "London,Paris" as a single city.

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to one destination city. Call this once per city; to price several cities, make one separate call for each.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "A single city that the customer wants to travel to, e.g. 'London'. Never a list of cities.",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": price_function}]
tools

## Chat, with tools

Same two adaptations as the day4 open source notebook: we check `message.tool_calls` rather than `finish_reason`, and we pass `temperature=0` - tool calling is a formatting task, not a creative one, and llama3.2 is markedly more consistent at 0. We also cap the tool loop so a confused model can't spin forever.

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments or "{}")
            city = arguments.get("destination_city")
            result = get_ticket_price(city)
        else:
            result = f"Error: there is no tool called {tool_call.function.name}"
        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })
    return responses

In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools, temperature=0)

    for _ in range(MAX_TOOL_ROUNDS):
        if not response.choices[0].message.tool_calls:
            break
        assistant_message = response.choices[0].message
        messages.append(assistant_message)
        messages.extend(handle_tool_calls(assistant_message))
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools, temperature=0)

    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## A bit more about what Gradio actually does:

1. Gradio constructs a frontend Svelte app based on our Python description of the UI
2. Gradio starts a server built upon the Starlette web framework listening on a free port that serves this Svelte app
3. Gradio creates backend routes for our callbacks, like chat(), which calls our functions

And of course when Gradio generates the frontend app, it ensures that the Submit button calls the right backend route.

That's it!

# Let's go multi-modal!

day5 uses `gpt-image-1-mini` at about 3 cents an image. We'll use **SD-Turbo** instead - an open weights Stable Diffusion distilled to produce a usable image in a *single* denoising step, which is what makes it viable without a GPU.

**No price alert this time.** Generate as many as you like; the only cost is your CPU's time.

The function below lazy-loads the pipeline on first call (the model downloads once, ~2.5GB, and is cached by HuggingFace afterwards). When `ENABLE_IMAGES` is False, or if `diffusers` isn't installed, it returns a drawn placeholder card instead - so the rest of the app is testable without waiting on diffusion.

In [ ]:
_pipe = None  # cached across calls - loading the pipeline is the expensive part


def _font(size):
    """A real font if we can find one, else PIL's tiny bitmap default."""
    for name in ("arial.ttf", "DejaVuSans.ttf", "Helvetica.ttc"):
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            continue
    return ImageFont.load_default()


def placeholder_image(city, note):
    """A drawn stand-in, so the UI still shows something when diffusion is off."""
    image = Image.new("RGB", (512, 512), (23, 32, 56))
    draw = ImageDraw.Draw(image)
    draw.rectangle([16, 16, 495, 495], outline=(90, 130, 200), width=3)
    for text, y, size, colour in [(city, 224, 40, (235, 240, 255)),
                                  (note, 286, 16, (140, 165, 210))]:
        font = _font(size)
        width = draw.textbbox((0, 0), text, font=font)[2]
        draw.text(((512 - width) / 2, y), text, fill=colour, font=font)
    return image


def artist(city):
    """Generate a vacation image for `city` with SD-Turbo, running locally on CPU."""
    global _pipe

    if not ENABLE_IMAGES:
        return placeholder_image(city, "set ENABLE_IMAGES = True for real images")

    if _pipe is None:
        try:
            import torch
            from diffusers import AutoPipelineForText2Image
        except ImportError:
            return placeholder_image(city, "pip install diffusers accelerate")

        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Loading sd-turbo on {device} (first run downloads ~2.5GB)...", flush=True)
        _pipe = AutoPipelineForText2Image.from_pretrained(
            "stabilityai/sd-turbo",
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        ).to(device)

    print(f"IMAGE TOOL CALLED: generating {city}", flush=True)
    prompt = (f"An image representing a vacation in {city}, showing tourist spots "
              f"and everything unique about {city}, in a vibrant pop-art style")
    # SD-Turbo is distilled for very few steps and needs guidance_scale=0.0
    return _pipe(prompt=prompt, num_inference_steps=1, guidance_scale=0.0).images[0]

In [ ]:
image = artist("New York City")
display(image)

## Talking, without an API

day5 calls `gpt-4o-mini-tts`. Windows already ships a speech synthesiser (SAPI) that works offline and needs no install at all, so we drive it through PowerShell and hand Gradio the resulting wav file.

It sounds noticeably more robotic than the OpenAI voices - that's the trade for free and offline. If you want open source *neural* TTS with far better quality, [Piper](https://github.com/rhasspy/piper) is the natural upgrade and still runs happily on CPU.

This cell is Windows-only; on Mac or Linux it degrades to silence rather than erroring.

In [ ]:
VOICE = "Microsoft Zira Desktop"   # try "Microsoft David Desktop" for a male voice
_audio_counter = count()


def talker(message):
    """Speak `message` with the offline Windows voice; returns a wav path for gr.Audio."""
    if sys.platform != "win32":
        print("talker(): Windows SAPI unavailable on this platform - skipping audio", flush=True)
        return None

    # A fresh filename each turn, so Gradio doesn't serve a cached clip
    path = os.path.join(tempfile.gettempdir(), f"flightai_{next(_audio_counter)}.wav")

    # Flatten newlines and escape quotes for the PowerShell single-quoted string
    text = " ".join(str(message).split()).replace("'", "''")

    script = (
        "Add-Type -AssemblyName System.Speech; "
        "$s = New-Object System.Speech.Synthesis.SpeechSynthesizer; "
        f"try {{ $s.SelectVoice('{VOICE}') }} catch {{ }}; "   # fall back to default voice
        f"$s.SetOutputToWaveFile('{path}'); "
        f"$s.Speak('{text}'); "
        "$s.Dispose()"
    )

    result = subprocess.run(
        ["powershell", "-NoProfile", "-NonInteractive", "-Command", script],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"talker() failed: {result.stderr.strip()}", flush=True)
        return None
    return path

In [ ]:
from IPython.display import Audio, display

wav = talker("Ticket price to Tokyo is 1420 dollars.")
display(Audio(wav, autoplay=False)) if wav else print("no audio")

## Let's bring this home:

1. A multi-modal AI assistant with image and audio generation
2. Tool calling with database lookup
3. A step towards an Agentic workflow

...and all of it running on your own machine.

In [ ]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments or "{}")
            city = arguments.get("destination_city")
            cities.append(city)
            result = get_ticket_price(city)
        else:
            result = f"Error: there is no tool called {tool_call.function.name}"
        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })
    return responses, cities

In [ ]:
def chat(history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools, temperature=0)
    cities = []
    image = None

    for _ in range(MAX_TOOL_ROUNDS):
        if not response.choices[0].message.tool_calls:
            break
        assistant_message = response.choices[0].message
        responses, new_cities = handle_tool_calls_and_return_cities(assistant_message)
        cities.extend(new_cities)
        messages.append(assistant_message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools, temperature=0)

    reply = response.choices[0].message.content
    history += [{"role": "assistant", "content": reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])

    return history, voice, image

## The 3 types of Gradio UI

`gr.Interface` is for standard, simple UIs

`gr.ChatInterface` is for standard ChatBot UIs

`gr.Blocks` is for custom UIs where you control the components and the callbacks

In [ ]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True)

# Exercises and Business Applications

Add more tools - perhaps to simulate actually booking a flight. The `set_ticket_price` function defined above is already sitting there unused; wiring it up as a second tool is a good five-minute exercise (the day4 open source notebook shows one way).

Then take this and apply it to your business. Because the whole stack is local, you can point it at data you would never send to a third-party API - which is often the real reason to run open source models rather than the cost.

### Notes on the open source swaps

- **Chat:** `llama3.2` is a 3B model. It handles this tool calling reliably at `temperature=0`, but stays chattier than the "no more than 1 sentence" instruction asks. `gpt-oss:20b` follows instructions better if you have the patience for it.
- **Images:** SD-Turbo at 1 step is the fastest sane option without a GPU. If you have CUDA, raise `num_inference_steps` to 4 and switch to `torch.float16` (already automatic above) for noticeably better images.
- **Audio:** SAPI is the zero-install choice. [Piper](https://github.com/rhasspy/piper) is the quality upgrade and is properly open source, at the cost of a pip install and a voice download.